In [1]:
import pandas as pd
from textblob import TextBlob
from sklearn.metrics import classification_report
import os

In [2]:
# 1. Step up one level from wherever Jupyter is currently looking
search_area = os.path.dirname(os.getcwd())
csv_path = None

# 2. Scan all nearby folders for the file
for root, dirs, files in os.walk(search_area):
    for file in files:
        if file.lower() == 'steam.csv': # This checks for both 'steam' and 'Steam'
            csv_path = os.path.join(root, file)
            break
    if csv_path:
        break

# 3. Load the data if it finds it
if csv_path:
    df = pd.read_csv(csv_path)

In [3]:
df.head()

,title,rating,review
0,Can't access the games I paid for,1,I opened tickets HT-2FQ9-CY9F-355R and HT-VJ6D...
1,"Great Service, they deserve a monopoly.",5,This must be at least 10 characters.
2,There is no service,1,There is no service. There is no help. There i...
3,I used Steam for years gaming on a PC…,2,I used Steam for years gaming on a PC until th...
4,I was forced to download Steam to log…,1,I was forced to download Steam to log into S&b...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2374 entries, 0 to 2373
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   title   2374 non-null   str  
 1   rating  2374 non-null   int64
 2   review  2374 non-null   str  
dtypes: int64(1), str(2)
memory usage: 55.8 KB


In [5]:
df['rating'].value_counts()

rating
1    1626
5     443
4     117
2     113
3      75
Name: count, dtype: int64

In [6]:
df['rating_sentiment'] = df['rating'].map({1:'negative',2:'negative',3:'neutral',4:'positive',5:'positive'})

In [7]:
df.head()

,title,rating,review,rating_sentiment
0,Can't access the games I paid for,1,I opened tickets HT-2FQ9-CY9F-355R and HT-VJ6D...,negative
1,"Great Service, they deserve a monopoly.",5,This must be at least 10 characters.,positive
2,There is no service,1,There is no service. There is no help. There i...,negative
3,I used Steam for years gaming on a PC…,2,I used Steam for years gaming on a PC until th...,negative
4,I was forced to download Steam to log…,1,I was forced to download Steam to log into S&b...,negative


In [8]:
df['rating_sentiment'].value_counts()

rating_sentiment
negative    1739
positive     560
neutral       75
Name: count, dtype: int64

In [9]:
df['full_text'] = df['title'] + ". " + df['review']

In [10]:
df['predicted_sentiment'] = df['full_text'].apply(lambda full_text: TextBlob(full_text).sentiment.polarity)

In [11]:
df.head()

,title,rating,review,rating_sentiment,full_text,predicted_sentiment
0,Can't access the games I paid for,1,I opened tickets HT-2FQ9-CY9F-355R and HT-VJ6D...,negative,Can't access the games I paid for. I opened ti...,-0.360000
1,"Great Service, they deserve a monopoly.",5,This must be at least 10 characters.,positive,"Great Service, they deserve a monopoly.. This ...",0.250000
2,There is no service,1,There is no service. There is no help. There i...,negative,There is no service. There is no service. Ther...,-0.433333
3,I used Steam for years gaming on a PC…,2,I used Steam for years gaming on a PC until th...,negative,I used Steam for years gaming on a PC…. I used...,-0.036029
4,I was forced to download Steam to log…,1,I was forced to download Steam to log into S&b...,negative,I was forced to download Steam to log…. I was ...,-0.096111


In [12]:
df['polarity'] = df['predicted_sentiment']

In [13]:
df.head()

,title,rating,review,rating_sentiment,full_text,predicted_sentiment,polarity
0,Can't access the games I paid for,1,I opened tickets HT-2FQ9-CY9F-355R and HT-VJ6D...,negative,Can't access the games I paid for. I opened ti...,-0.360000,-0.360000
1,"Great Service, they deserve a monopoly.",5,This must be at least 10 characters.,positive,"Great Service, they deserve a monopoly.. This ...",0.250000,0.250000
2,There is no service,1,There is no service. There is no help. There i...,negative,There is no service. There is no service. Ther...,-0.433333,-0.433333
3,I used Steam for years gaming on a PC…,2,I used Steam for years gaming on a PC until th...,negative,I used Steam for years gaming on a PC…. I used...,-0.036029,-0.036029
4,I was forced to download Steam to log…,1,I was forced to download Steam to log into S&b...,negative,I was forced to download Steam to log…. I was ...,-0.096111,-0.096111


In [14]:
df['predicted_sentiment'] = df['predicted_sentiment'].apply(lambda p: 'positive' if p>0.1 else ('negative') if p<-0.1 else 'neutral')

In [15]:
textblob_accuracy = (df['rating_sentiment'] == df['predicted_sentiment']).mean()

In [16]:
textblob_accuracy

np.float64(0.5501263689974726)

In [17]:
df['predicted_sentiment'].value_counts()

predicted_sentiment
negative    886
neutral     795
positive    693
Name: count, dtype: int64

In [18]:
# We compare the ACTUAL ratings vs. TextBlob's GUESSES
actual_labels = df['rating_sentiment']
predicted_labels = df['predicted_sentiment']

In [19]:
# Generate and print the full report
report = classification_report(actual_labels, predicted_labels)
print(report)

              precision    recall  f1-score   support

    negative       0.95      0.49      0.64      1739
     neutral       0.04      0.43      0.07        75
    positive       0.62      0.77      0.68       560

    accuracy                           0.55      2374
   macro avg       0.54      0.56      0.47      2374
weighted avg       0.85      0.55      0.64      2374

